# [DS-02] GHSA 변화 이벤트 및 보조 신호 종합 분석 Notebook

> **작성팀**: Watchback Data Science (DS) Team  
> **분석 대상**: 최근 1년치 공식 Reviewed GHSA 10,616건 및 과거 6개월 스냅샷 2,546건  
> **목적**: GHSA 속성(Severity, 영향 버전, 패치 정보)의 통계적 유효성을 입증하고, EPSS 위험 변화와의 관계 및 보조 신호(B/C축)를 정의함.

## 1. 환경 설정 및 데이터셋 로드

- **현재 1년치 데이터**: `data/ghsa_events_1year.json` (10,616건)
- **과거 6개월 스냅샷**: `C:\Users\EL021\Downloads\ghsa_reviewed_published_2025-08-20_to_2026-02-20.json` (2,546건)
- 현재 1년치 데이터는 api호출로 집계. 6개월 스냅샷은 공식 GHSA repository에 6개월전 commit 날짜를 기준으로 6개월 간의 데이터를 가져왔음을 알림.

In [2]:
import json
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path
import pandas as pd

# 데이터 로드
curr_path = Path("../data/ghsa_events_1year.json")
past_path = Path(r"C:\Users\EL021\Downloads\ghsa_reviewed_published_2025-08-20_to_2026-02-20.json")

with open(curr_path, "r", encoding="utf-8") as f:
    curr_data = json.load(f)

past_data = []
if past_path.exists():
    p_raw = json.load(open(past_path, "r", encoding="utf-8"))
    past_data = p_raw.get("advisories", []) if isinstance(p_raw, dict) else p_raw

print(f"✓ 현재 1년치 GHSA 데이터 로드 완료: {len(curr_data):,}건")
print(f"✓ 과거 6개월 스냅샷 데이터 로드 완료: {len(past_data):,}건")

✓ 현재 1년치 GHSA 데이터 로드 완료: 10,616건
✓ 과거 6개월 스냅샷 데이터 로드 완료: 2,546건


## 2. CVE 보유 여부 및 No-CVE 시계열 지연(Time Lag) 분석

- 1년치 데이터 중 공식 CVE 식별자가 없는 취약점 비율 확인
- 최근 월별로 No-CVE 비율이 어떻게 변화하는지 시계열 비교 (Time Lag 실증)

In [3]:
total = len(curr_data)
no_cve = [x for x in curr_data if not x.get("cve_id")]
has_cve = [x for x in curr_data if x.get("cve_id")]

print(f"[CVE 보유 현황]")
print(f"• CVE 보유: {len(has_cve):,}건 ({len(has_cve)/total*100:.2f}%)")
print(f"• CVE 미보유 (No-CVE): {len(no_cve):,}건 ({len(no_cve)/total*100:.2f}%)")

# 월별 No-CVE 비율 추이
all_monthly = Counter(datetime.fromisoformat(x['published_at'].replace('Z', '+00:00')).strftime('%Y-%m') for x in curr_data if x.get('published_at'))
no_cve_monthly = Counter(datetime.fromisoformat(x['published_at'].replace('Z', '+00:00')).strftime('%Y-%m') for x in no_cve if x.get('published_at'))

time_df = pd.DataFrame([
    {"월(Year-Month)": ym, "전체 건수": all_monthly[ym], "No-CVE 건수": no_cve_monthly[ym], "No-CVE 비율(%)": f"{no_cve_monthly[ym]/all_monthly[ym]*100:.2f}%"}
    for ym in sorted(all_monthly.keys())
])
time_df

[CVE 보유 현황]
• CVE 보유: 8,951건 (84.32%)
• CVE 미보유 (No-CVE): 1,665건 (15.68%)


,월(Year-Month),전체 건수,No-CVE 건수,No-CVE 비율(%)
0,2025-08,186,35,18.82%
1,2025-09,368,39,10.60%
2,2025-10,384,49,12.76%
3,2025-11,253,28,11.07%
4,2025-12,376,41,10.90%
5,2026-01,491,47,9.57%
6,2026-02,765,83,10.85%
7,2026-03,1639,294,17.94%
8,2026-04,1582,244,15.42%
9,2026-05,1701,230,13.52%


## 3. 과거 스냅샷 vs 현재 데이터셋 매칭: CVE 신규 부여(Backfill) 전수 조사

- 6개월 전 No-CVE였던 취약점 중 현재 시점에 CVE가 신규 발급된 비율 확인
- CVE 발급 후 EPSS 점수가 활성화된 현황 검증

In [4]:
curr_map = {x["ghsa_id"]: x for x in curr_data if x.get("ghsa_id")}

past_no_cves = []
for p in past_data:
    aliases = p.get("aliases", []) or []
    cves = [a for a in aliases if a.startswith("CVE-")]
    if not cves:
        past_no_cves.append(p)

transitions = []
still_no = []

for p in past_no_cves:
    gid = p.get("id") or p.get("ghsa_id")
    c = curr_map.get(gid)
    if c and c.get("cve_id"):
        transitions.append({
            "ghsa_id": gid,
            "new_cve_id": c.get("cve_id"),
            "severity": c.get("severity"),
            "epss": c.get("epss", {}).get("percentage") if c.get("epss") else None,
            "published_at": p.get("published") or p.get("published_at"),
            "updated_at": c.get("updated_at")
        })
    else:
        still_no.append(p)

print(f"• 과거 시점 No-CVE 대상: {len(past_no_cves):,}건")
print(f"• 현재 시점 CVE '신규 부여(생성)' 건수: {len(transitions):,}건 ({len(transitions)/len(past_no_cves)*100:.2f}%)")
print(f"• 여전히 CVE가 없는 건수: {len(still_no):,}건 ({len(still_no)/len(past_no_cves)*100:.2f}%)")

trans_df = pd.DataFrame(transitions[:5])
trans_df

• 과거 시점 No-CVE 대상: 341건
• 현재 시점 CVE '신규 부여(생성)' 건수: 56건 (16.42%)
• 여전히 CVE가 없는 건수: 285건 (83.58%)


,ghsa_id,new_cve_id,severity,epss,published_at,updated_at
0,GHSA-3vg9-h568-4w9m,CVE-2025-71354,high,0.00253,2025-08-26T18:36:42Z,2026-07-21T13:14:11Z
1,GHSA-6w4w-5w54-rjvr,CVE-2025-71358,high,0.00431,2025-08-26T18:37:10Z,2026-07-18T17:26:56Z
2,GHSA-8r4j-24qv-fmq9,CVE-2025-71361,high,0.00339,2025-08-26T18:38:43Z,2026-07-20T21:31:33Z
3,GHSA-xp4f-hrf8-rxw7,CVE-2025-71344,high,0.00638,2025-08-26T21:34:37Z,2026-07-21T19:24:21Z
4,GHSA-pfrf-9r5f-73f5,CVE-2026-29067,high,0.00322,2025-12-08T22:19:38Z,2026-03-09T15:47:06Z


## 4. GHSA Severity 등급별 EPSS 분포 및 통계 분석

- GitHub Advisory Severity 등급(Critical, High, Medium, Low)에 따른 실제 악용 확률(EPSS)의 분포를 분석합니다.
- B축 배점에서 Critical에 최고 점수(+25)를 부여하는 통계적 타당성을 입증합니다.

In [5]:
sev_stats = defaultdict(list)

for item in curr_data:
    sev = item.get("severity", "unknown").capitalize()
    epss_obj = item.get("epss") or {}
    epss_score = epss_obj.get("percentage")
    if epss_score is not None:
        sev_stats[sev].append(float(epss_score))

rows = []
for sev in ["Critical", "High", "Medium", "Low"]:
    scores = sev_stats[sev]
    n = len(scores)
    if n == 0:
        continue
    avg_epss = sum(scores) / n
    med_epss = pd.Series(scores).median()
    high_risk_ratio = sum(1 for s in scores if s >= 0.10) / n
    top_risk_ratio = sum(1 for s in scores if s >= 0.30) / n
    rows.append({
        "Severity 등급": sev,
        "건수": f"{n:,}건",
        "비중": f"{n/total*100:.2f}%",
        "EPSS 평균": f"{avg_epss*100:.2f}%",
        "EPSS 중앙값": f"{med_epss*100:.2f}%",
        "고위험군 비율 (EPSS >= 10%)": f"{high_risk_ratio*100:.2f}%",
        "최상위 고위험군 (EPSS >= 30%)": f"{top_risk_ratio*100:.2f}%"
    })

sev_df = pd.DataFrame(rows)
sev_df

,Severity 등급,건수,비중,EPSS 평균,EPSS 중앙값,고위험군 비율 (EPSS >= 10%),최상위 고위험군 (EPSS >= 30%)
0,Critical,916건,8.63%,3.03%,0.51%,5.57%,2.84%
1,High,"3,243건",30.55%,0.90%,0.37%,1.26%,0.46%
2,Medium,"3,641건",34.30%,0.36%,0.26%,0.16%,0.03%
3,Low,765건,7.21%,0.31%,0.24%,0.13%,0.00%


## 5. 4대 핵심 교차 케이스 분석 (Severity × CVSS Score)

- B축 평가 규칙의 4대 교차 케이스별 위험 분포를 산출합니다.
  - **Case 1**: `Critical & CVSS >= 9.0` (`R-GHSA-SEV-STRONG`, +25점)
  - **Case 2**: `Critical & CVSS < 9.0` (`R-GHSA-SEV-SINGLE`, +15점)
  - **Case 3**: `Not Critical & CVSS >= 9.0` (`R-GHSA-SEV-SINGLE`, +15점)
  - **Case 4**: `High & 8.0 <= CVSS < 9.0` (`R-GHSA-SEV-MODERATE`, +5점)
  - **Case 5**: 기타 Medium/Low/CVSS<8.0 (0점, 노이즈 구간)

In [6]:
case_lists = defaultdict(list)

for item in curr_data:
    sev = (item.get("severity") or "").lower()
    cvss = float(item.get("cvss", {}).get("score") or 0.0)
    epss = float(item.get("epss", {}).get("percentage") or 0.0) if item.get("epss") else 0.0
    
    if sev == "critical" and cvss >= 9.0:
        case_lists["Case 1 (Critical & CVSS >= 9.0)"].append(epss)
    elif sev == "critical" and cvss < 9.0:
        case_lists["Case 2 (Critical & CVSS < 9.0)"].append(epss)
    elif sev != "critical" and cvss >= 9.0:
        case_lists["Case 3 (Not Critical & CVSS >= 9.0)"].append(epss)
    elif sev != "critical" and 8.0 <= cvss < 9.0:
        case_lists["Case 4 (High & 8.0 <= CVSS < 9.0)"].append(epss)
    else:
        case_lists["Case 5 (Others / Noise)"].append(epss)

case_rows = []
for c_name in sorted(case_lists.keys()):
    eps = case_lists[c_name]
    cnt = len(eps)
    avg_val = sum(eps) / cnt if cnt > 0 else 0.0
    high_pct = sum(1 for e in eps if e >= 0.10) / cnt if cnt > 0 else 0.0
    case_rows.append({
        "케이스 구분": c_name,
        "건수": f"{cnt:,}건",
        "비중": f"{cnt/total*100:.2f}%",
        "EPSS 평균": f"{avg_val*100:.2f}%",
        "고위험군 비율 (EPSS >= 10%)": f"{high_pct*100:.2f}%"
    })

case_df = pd.DataFrame(case_rows)
case_df

,케이스 구분,건수,비중,EPSS 평균,고위험군 비율 (EPSS >= 10%)
0,Case 1 (Critical & CVSS >= 9.0),787건,7.41%,2.97%,5.46%
1,Case 2 (Critical & CVSS < 9.0),318건,3.00%,1.39%,2.52%
2,Case 3 (Not Critical & CVSS >= 9.0),80건,0.75%,0.81%,2.50%
3,Case 4 (High & 8.0 <= CVSS < 9.0),"1,117건",10.52%,0.93%,1.61%
4,Case 5 (Others / Noise),"8,314건",78.32%,0.41%,0.34%


## 6. CVSS 버전별 (v3.1 vs v4.0 vs 둘 다) 보유 현황 분석

- 10,616건에 대한 CVSS 점수 보유율 및 세부 버전(v3.1, v4.0) 분포를 확인합니다.

In [7]:
both_valid = 0
v3_only_valid = 0
v4_only_valid = 0
none_valid = 0

for item in curr_data:
    sev = item.get("cvss_severities") or {}
    v3 = sev.get("cvss_v3") or {}
    v4 = sev.get("cvss_v4") or {}
    
    v3_score = v3.get("score")
    v4_score = v4.get("score")
    top_score = item.get("cvss", {}).get("score")
    top_vec = item.get("cvss", {}).get("vector_string") or ""
    
    is_v3 = (v3_score is not None and v3_score > 0) or ("CVSS:3." in top_vec) or (top_score is not None and not "CVSS:4." in top_vec)
    is_v4 = (v4_score is not None and v4_score > 0) or ("CVSS:4." in top_vec)
    
    if is_v3 and is_v4:
        both_valid += 1
    elif is_v3:
        v3_only_valid += 1
    elif is_v4:
        v4_only_valid += 1
    else:
        none_valid += 1

cvss_df = pd.DataFrame([
    {"구분": "CVSS v3.x 만 유효 보유", "건수": f"{v3_only_valid:,}건", "비율": f"{v3_only_valid/total*100:.2f}%"},
    {"구분": "CVSS v4.0 만 유효 보유", "건수": f"{v4_only_valid:,}건", "비율": f"{v4_only_valid/total*100:.2f}%"},
    {"구분": "v3.x + v4.0 둘 다(Both) 보유", "건수": f"{both_valid:,}건", "비율": f"{both_valid/total*100:.2f}%"},
    {"구분": "CVSS 점수 결측 (None)", "건수": f"{none_valid:,}건", "비율": f"{none_valid/total*100:.2f}%"}
])
cvss_df

,구분,건수,비율
0,CVSS v3.x 만 유효 보유,"5,990건",56.42%
1,CVSS v4.0 만 유효 보유,"3,031건",28.55%
2,v3.x + v4.0 둘 다(Both) 보유,"1,469건",13.84%
3,CVSS 점수 결측 (None),126건,1.19%


## 7. 패치 가용성(Patch Actionability) 및 대표 고위험 알림 사례 도출

- 1년치 데이터셋에서 패치 보유 여부(`patched` vs `no_patch`)를 산출합니다.
- EPSS 90% 이상의 최상위 악용 위험과 패치 출시가 결합된 실제 대표 사례 5선을 추출합니다.

In [9]:
patched_cnt = sum(1 for x in curr_data if x.get("vulnerabilities", [{}])[0].get("first_patched_version"))
no_patch_cnt = total - patched_cnt

print(f"• 패치 존재 (즉시 조치 가능): {patched_cnt:,}건 ({patched_cnt/total*100:.2f}%)")
print(f"• 미패치 (No Fix 상태): {no_patch_cnt:,}건 ({no_patch_cnt/total*100:.2f}%)")

# 대표 사례 추출
high_samples = []
for item in curr_data:
    epss = float(item.get("epss", {}).get("percentage") or 0.0) if item.get("epss") else 0.0
    if epss >= 0.70:
        vulns = item.get("vulnerabilities", [])
        pkg = vulns[0].get("package", {}).get("name") if vulns else "unknown"
        eco = vulns[0].get("package", {}).get("ecosystem") if vulns else "unknown"
        v_range = vulns[0].get("vulnerable_version_range", "") if vulns else ""
        first_patched = vulns[0].get("first_patched_version") if vulns else None
        if isinstance(first_patched, dict):
            patch = first_patched.get("identifier") or "None"
        else:
            patch = first_patched or "None"
        high_samples.append({
            "CVE ID": item.get("cve_id"),
            "GHSA ID": item.get("ghsa_id"),
            "패키지명 (생태계)": f"{pkg} ({eco})",
            "Severity": item.get("severity"),
            "EPSS 점수": f"{epss*100:.2f}%",
            "취약 버전 범위 ➔ 패치 버전": f"{v_range} ➔ {patch}"
        })

high_samples.sort(key=lambda x: float(x["EPSS 점수"].replace("%", "")), reverse=True)
sample_df = pd.DataFrame(high_samples[:5])
sample_df

• 패치 존재 (즉시 조치 가능): 9,279건 (87.41%)
• 미패치 (No Fix 상태): 1,337건 (12.59%)


,CVE ID,GHSA ID,패키지명 (생태계),Severity,EPSS 점수,취약 버전 범위 ➔ 패치 버전
0,CVE-2026-33017,GHSA-vwmf-pq79-vjvx,langflow (pip),critical,99.84%,<= 1.8.2 ➔ 1.9.0
1,CVE-2025-55182,GHSA-fv66-9v8q-g76r,react-server-dom-webpack (npm),critical,99.62%,">= 19.1.0, < 19.1.2 ➔ 19.1.2"
2,CVE-2025-68613,GHSA-v98v-ff95-f3cp,n8n (npm),critical,97.88%,">= 0.211.0, < 1.120.4 ➔ 1.120.4"
3,CVE-2026-34197,GHSA-rxpj-7qvf-xv32,org.apache.activemq:activemq-broker (maven),high,97.22%,< 5.19.5 ➔ 5.19.5
4,CVE-2025-54236,GHSA-wh92-6q6g-px7j,magento/community-edition (composer),critical,96.74%,<= 2.4.5-p14 ➔ None


## 8. 최종 결론 및 보조 신호 정의 요약

1. **B축 (현재 심각도)**: `Critical` 등급은 `Medium/Low` 대비 고위험 악용 비율이 35~40배 높으므로 최고 배점(+25점) 부여 타당성 입증.
2. **C축 (조치 가능성 변화)**: 전체 취약점의 11.7%가 최초 공개 시 미패치 상태이며, 신규 패치 출시(`R-GHSA-PATCH-NEW`) 시 조치 가능성 신호(+20점)로 발화.
3. **CVE 발급 지연(Time Lag)**: 공개 초기 약 15.7%는 CVE가 없으므로 초기에는 GHSA 심각도/패치 정보로 모니터링하다가 CVE 발급 시 EPSS/KEV 추적 모드로 자동 전환.